# TTS 디버그 — aṅguli 1개만 생성

변환 과정 전체를 출력하여 어디서 문제가 발생하는지 확인

In [ ]:
!pip install -q transformers torch torchaudio soundfile
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

In [ ]:
from transformers import VitsModel, AutoTokenizer
import soundfile as sf
import IPython.display as ipd

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = VitsModel.from_pretrained('facebook/mms-tts-hin').to(device)
tokenizer = AutoTokenizer.from_pretrained('facebook/mms-tts-hin')
sr = model.config.sampling_rate
print(f'모델 로드 완료 | sample_rate: {sr}')

In [ ]:
# === 데바나가리 변환 함수 ===
CONSONANTS = {
    'kh': '\u0916', 'k': '\u0915', 'gh': '\u0918', 'g': '\u0917', '\u1e45': '\u0919',
    'ch': '\u091b', 'c': '\u091a', 'jh': '\u091d', 'j': '\u091c', '\u00f1': '\u091e',
    '\u1e6dh': '\u0920', '\u1e6d': '\u091f', '\u1e0dh': '\u0922', '\u1e0d': '\u0921', '\u1e47': '\u0923',
    'th': '\u0925', 't': '\u0924', 'dh': '\u0927', 'd': '\u0926', 'n': '\u0928',
    'ph': '\u092b', 'p': '\u092a', 'bh': '\u092d', 'b': '\u092c', 'm': '\u092e',
    'y': '\u092f', 'r': '\u0930', 'l': '\u0932', '\u1e37': '\u0933',
    'v': '\u0935', 's': '\u0938', 'h': '\u0939',
}
VOWELS_IND = { '\u0101': '\u0906', 'a': '\u0905', '\u012b': '\u0908', 'i': '\u0907', '\u016b': '\u090a', 'u': '\u0909', 'e': '\u090f', 'o': '\u0913' }
VOWELS_DEP = { '\u0101': '\u093e', 'a': '', '\u012b': '\u0940', 'i': '\u093f', '\u016b': '\u0942', 'u': '\u0941', 'e': '\u0947', 'o': '\u094b' }
VIRAMA = '\u094d'

def pali_to_devanagari(roman):
    result = ''
    i = 0
    s = roman.lower()
    while i < len(s):
        ch = s[i]
        if ch in ' ,.;:!?-\n\r\t': result += ch; i += 1; continue
        if ch == '\u1e43': result += '\u0902'; i += 1; continue
        three = s[i:i+3]; two = s[i:i+2]
        consonant = None; consumed = 0
        if three in CONSONANTS: consonant = CONSONANTS[three]; consumed = 3
        elif two in CONSONANTS: consonant = CONSONANTS[two]; consumed = 2
        elif ch in CONSONANTS: consonant = CONSONANTS[ch]; consumed = 1
        if consonant:
            i += consumed
            if i < len(s) and s[i] in VOWELS_IND: result += consonant + VOWELS_DEP[s[i]]; i += 1
            else: result += consonant + VIRAMA
            continue
        if ch in VOWELS_IND: result += VOWELS_IND[ch]; i += 1; continue
        result += ch; i += 1
    return result

print('변환 함수 준비 완료')

In [ ]:
# === 디버그: aṅguli 단계별 확인 ===
word = 'aṅguli'
print(f'1. 원문: {word}')
print(f'   글자: {[c for c in word]}')
print(f'   유니코드: {[hex(ord(c)) for c in word]}')
print()

deva = pali_to_devanagari(word)
print(f'2. 데바나가리: {deva}')
print(f'   글자: {[c for c in deva]}')
print(f'   유니코드: {[hex(ord(c)) for c in deva]}')
print()

# 기대값과 비교
expected = '\u0905\u0919\u094d\u0917\u0941\u0932\u093f'  # अङ्गुलि
print(f'3. 기대값: {expected}')
print(f'   일치: {deva == expected}')
print()

# 토크나이저
tokens = tokenizer(deva, return_tensors='pt')
print(f'4. 토크나이저 input_ids: {tokens.input_ids}')
print(f'   토큰 수: {tokens.input_ids.shape}')
print()

# 비교: 힌디어 단어 '아굴리' (손가락)
hindi_word = 'अंगुली'  # 힌디어 표준 표기
hindi_tokens = tokenizer(hindi_word, return_tensors='pt')
print(f'5. 힌디어 비교: {hindi_word}')
print(f'   토크나이저 input_ids: {hindi_tokens.input_ids}')
print(f'   토큰 수: {hindi_tokens.input_ids.shape}')

In [ ]:
# === 음성 3개 생성: 변환된 것 + 힌디어 표기 + 로마자 직접 ===
import numpy as np

def generate_audio(text, label):
    print(f'\n--- {label}: "{text}" ---')
    inputs = tokenizer(text, return_tensors='pt').to(device)
    print(f'  input_ids: {inputs.input_ids.cpu()}')
    with torch.no_grad():
        output = model(**inputs).waveform
    audio = output.squeeze().cpu().numpy()
    duration = len(audio) / sr
    print(f'  길이: {duration:.2f}초')
    return audio

# 1) 변환된 데바나가리
audio1 = generate_audio(deva, '변환 결과')

# 2) 힌디어 표준 표기
audio2 = generate_audio('अंगुली', '힌디어 표기 (अंगुली)')

# 3) 다른 변형
audio3 = generate_audio('अङ्गुलि', '빠알리 정확 표기 (अङ्गुलि)')

print('\n=== 아래에서 각각 재생하여 비교 ===' )

In [ ]:
# 재생: 변환 결과
print(f'재생 1: 변환 결과 ({deva})')
ipd.display(ipd.Audio(audio1, rate=sr))

In [ ]:
# 재생: 힌디어 표기
print('재생 2: 힌디어 표기 (अंगुली)')
ipd.display(ipd.Audio(audio2, rate=sr))

In [ ]:
# 재생: 빠알리 정확 표기
print('재생 3: 빠알리 정확 표기 (अङ्गुलि)')
ipd.display(ipd.Audio(audio3, rate=sr))

In [ ]:
# === 추가 테스트: buddha, dhammaṃ, bhikkhu ===
test_words = ['buddha', 'dhammaṃ', 'bhikkhu', 'nibbāna', 'mettā']
for w in test_words:
    d = pali_to_devanagari(w)
    audio = generate_audio(d, f'{w} → {d}')
    print(f'  재생:')
    ipd.display(ipd.Audio(audio, rate=sr))